In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers

In [4]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

In [6]:
!pip install torch torchvision matplotlib
# Check available GPU
!nvidia-smi

Sat Apr  4 10:07:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   39C    P0             56W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
import json
from google.colab import files

# --- Upload pre-built high-quality SFT data ---
# train_sft.json is generated by build_sft_data.py from API results
# (reference + multi_agent_api + multi_agent_review) paired with topic source texts.
uploaded = files.upload()  # select train_sft.json

with open("train_sft.json") as f:
    sft_data = json.load(f)

print(f"SFT samples: {len(sft_data)}")
total_turns = sum(len([m for m in s["messages"] if m["role"] != "system"]) for s in sft_data)
print(f"Total dialogue turns: {total_turns}")
print(f"\nSample #1 system prompt (first 200 chars):")
print(sft_data[0]["messages"][0]["content"][:200] + "...")
print(f"\nSample #1 first exchange:")
for m in sft_data[0]["messages"][1:3]:
    print(f"  [{m['role']}]: {m['content'][:120]}...")

Saving train_sft.json to train_sft (1).json
SFT samples: 26
Total dialogue turns: 214

Sample #1 system prompt (first 200 chars):
You are a podcast script writer. Given source material, write a natural two-person podcast conversation between a Host and an Expert.

Topic: AI Bias and Fairness

Source material:
Algorithmic bias ha...

Sample #1 first exchange:
  [user]: Today we're talking about something that sounds like a technical problem but is really a human one: algorithmic bias. Th...
  [assistant]: Oh, it's so much more insidious than that. Think of it this way: if you're baking bread, you don't need to add prejudice...


In [8]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import torch

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")

if "A100" in gpu_name or "H100" in gpu_name:
    MODEL_NAME = "unsloth/Llama-3.1-8B-Instruct"
    BATCH_SIZE = 4
elif "L4" in gpu_name or "T4" in gpu_name:
    MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
    BATCH_SIZE = 2
else:
    MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
    BATCH_SIZE = 2

print(f"Using model: {MODEL_NAME}, batch_size: {BATCH_SIZE}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

# ✅ 换成这个
dataset = load_dataset("json", data_files="train_sft.json", split="train")

def to_text(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(to_text, batched=True, remove_columns=dataset.column_names)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",      # 告诉 Trainer 用 "text" 这列
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        output_dir="./output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_steps=5,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        save_strategy="epoch",
        optim="adamw_8bit",
        seed=42,
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"Training complete. Final loss: {stats.training_loss:.4f}")

/tmp/ipykernel_4567/657113194.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: NVIDIA A100-SXM4-80GB
Using model: unsloth/Llama-3.1-8B-Instruct, batch_size: 4
==((====))==  Unsloth 2026.4.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/26 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26 | Num Epochs = 3 | Total steps = 6
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.131979
2,2.182405
3,2.175098
4,2.087932
5,2.110936
6,2.021779


Training complete. Final loss: 2.1184


In [9]:
# Quick sanity-check: pick 2 samples from training data, use only the
# system prompt + first host turn, and see if the model produces a
# reasonable expert response.
import json

with open("train_sft.json") as f:
    test_samples = json.load(f)[:2]

FastLanguageModel.for_inference(model)

for i, sample in enumerate(test_samples):
    msgs = sample["messages"]
    sys_msg = msgs[0]
    first_host = msgs[1]  # first user (host) turn

    messages = [sys_msg, first_host]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    input_ids = inputs["input_ids"]

    outputs = model.generate(
        input_ids,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
    )
    response = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True,
    )

    topic_line = [l for l in sys_msg["content"].split("\n") if l.startswith("Topic:")][0]
    print(f"\n{'='*60}")
    print(f"[Sample {i+1}] {topic_line}")
    print(f"Host: {first_host['content'][:150]}...")
    print(f"Expert: {response}")
    print("-" * 60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn


[Sample 1] Topic: AI Bias and Fairness
Host: Today we're talking about something that sounds like a technical problem but is really a human one: algorithmic bias. These systems are making decisio...
Expert: That's a common misconception. Bias in AI isn't just about a programmer's personal prejudices or intentional discrimination. It's more complex and insidious than that. Bias can arise from the data itself, the way it's collected, labeled, and represented. It can also come from the algorithms and models that process that data, even if they're designed to be neutral.

Think of it like a mirror that reflects the world around it. If the world is already biased, the mirror will reflect those biases. For example, if a hiring model is trained on a company's historical employment data and that company has historically favored male candidates, the model will learn to replicate that preference, even if it's not because of any personal prejudice on the part of the programmer. The data itself i

In [10]:
from google.colab import files

# 保存 LoRA adapter
model.save_pretrained("podcast-expert-lora")
tokenizer.save_pretrained("podcast-expert-lora")

# 打包下载
!zip -r podcast-expert-lora.zip podcast-expert-lora/
files.download("podcast-expert-lora.zip")

print("Done. Check your Downloads folder.")

  adding: podcast-expert-lora/ (stored 0%)
  adding: podcast-expert-lora/adapter_config.json (deflated 58%)
  adding: podcast-expert-lora/README.md (deflated 65%)
  adding: podcast-expert-lora/adapter_model.safetensors (deflated 10%)
  adding: podcast-expert-lora/tokenizer.json (deflated 85%)
  adding: podcast-expert-lora/chat_template.jinja (deflated 72%)
  adding: podcast-expert-lora/tokenizer_config.json (deflated 45%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done. Check your Downloads folder.
